In [7]:
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from typing import TypedDict, Dict, List, Literal
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolNode


# Define get_weather BEFORE referencing it in tools list
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    if location.lower() == "new york":
        return "The weather in New York is cloudy with a high of 20°C."
    elif location.lower() == "san francisco":
        return "The weather in San Francisco is foggy with a high of 18°C."
    return "Sorry, I don't have weather information for that location."


tools = [get_weather]

llm = ChatOpenAI(
    api_key="ollama",
    model="qwen2.5",
    base_url="http://localhost:11434/v1",
).bind_tools(tools)


async def node_llm(state: MessagesState) -> MessagesState:
    response = await llm.ainvoke(state["messages"])
    return {"messages": [response]}

def router(state: MessagesState) -> Literal["tools", END]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END


graph = StateGraph(state_schema=MessagesState)

node_tools = ToolNode(tools)

graph.add_node("llm", node_llm)
graph.add_node("tools", node_tools)

graph.add_edge(START, "llm")
graph.add_conditional_edges("llm", router)
graph.add_edge("tools", "llm")

memory_saver = MemorySaver()

graph_compiled = graph.compile(checkpointer=memory_saver)

state = {"messages": [HumanMessage(content="What's the weather in New York?")]}
config = {"configurable": {"thread_id": 1}}

await graph_compiled.ainvoke(state, config=config)


{'messages': [HumanMessage(content="What's the weather in New York?", additional_kwargs={}, response_metadata={}, id='20b9da5e-390d-4218-8916-ce07def68b7f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 155, 'total_tokens': 176, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen2.5', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-158', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e7db3-be90-79d2-b070-0f1a61f5139d-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_kndihcb8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 155, 'output_tokens': 21, 'total_tokens': 176, 'input_token_details': {}, 'output_token_details': {}}),
  ToolMessage(content='The weather in New York is cloudy with a high of 20°C.', name='get_weather', id='b2596834-a872-4ae8-

In [8]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

# Overall state — everything the graph tracks internally
class OverallState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]  # reducer: always appends
    question: str
    answer: str

# Input state — only what the caller provides
class InputState(TypedDict):
    question: str

# Output state — only what the caller receives back
class OutputState(TypedDict):
    answer: str


llm_plain = ChatOpenAI(
    api_key="ollama",
    model="qwen2.5",
    base_url="http://localhost:11434/v1",
)

async def node_answer(state: OverallState) -> dict:
    human_msg = HumanMessage(content=state["question"])
    response = await llm_plain.ainvoke([human_msg])
    return {
        "messages": [human_msg, response], 
        "answer": response.content,
    }


qa_graph = StateGraph(OverallState, input=InputState, output=OutputState)
qa_graph.add_node("answer", node_answer)
qa_graph.add_edge(START, "answer")
qa_graph.add_edge("answer", END)

qa_compiled = qa_graph.compile()

await qa_compiled.ainvoke({"question": "What is the capital of France?"})


/var/folders/gn/mdrgn38s3lzddxysxfdzn12m0000gq/T/ipykernel_31734/235994564.py:34: LangGraphDeprecatedSinceV05: `input` is deprecated and will be removed. Please use `input_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  qa_graph = StateGraph(OverallState, input=InputState, output=OutputState)
/var/folders/gn/mdrgn38s3lzddxysxfdzn12m0000gq/T/ipykernel_31734/235994564.py:34: LangGraphDeprecatedSinceV05: `output` is deprecated and will be removed. Please use `output_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  qa_graph = StateGraph(OverallState, input=InputState, output=OutputState)


{'answer': 'The capital of France is Paris.'}

In [ ]:
# ─── Multi-Agent System with Subgraphs + Structured Output ───────────────────
# Flow: User Question → Classifier Agent → Billing Agent OR Technical Agent

from langchain_core.messages import SystemMessage
from pydantic import BaseModel

llm_agent = ChatOpenAI(
    api_key="ollama",
    model="qwen2.5",
    base_url="http://localhost:11434/v1",
)

# ─── Structured output model for classifier ───────────────────────────────────
class Category(BaseModel):
    category: Literal["billing", "technical"]

llm_classifier = llm_agent.with_structured_output(Category)


# ─── Parent State ─────────────────────────────────────────────────────────────
class ParentState(TypedDict):
    question: str
    category: str   # filled by classifier agent
    answer: str     # filled by billing or technical agent


# ─── Classifier Subgraph ──────────────────────────────────────────────────────
class ClassifierState(TypedDict):
    question: str   # shared with parent
    category: str   # shared with parent

async def classify_node(state: ClassifierState) -> dict:
    response = await llm_classifier.ainvoke([
        SystemMessage(content="Classify the user question as billing or technical."),
        HumanMessage(content=state["question"]),
    ])
    return {"category": response.category} 

classifier_graph = StateGraph(ClassifierState)
classifier_graph.add_node("classify", classify_node)
classifier_graph.add_edge(START, "classify")
classifier_graph.add_edge("classify", END)
classifier_agent = classifier_graph.compile()


# ─── Billing Subgraph ─────────────────────────────────────────────────────────
class BillingState(TypedDict):
    question: str   # shared with parent
    answer: str     # shared with parent

async def billing_node(state: BillingState) -> dict:
    response = await llm_agent.ainvoke([
        SystemMessage(content="You are a billing support agent. Answer the user's billing question."),
        HumanMessage(content=state["question"]),
    ])
    return {"answer": response.content}

billing_graph = StateGraph(BillingState)
billing_graph.add_node("billing", billing_node)
billing_graph.add_edge(START, "billing")
billing_graph.add_edge("billing", END)
billing_agent = billing_graph.compile()


# ─── Technical Subgraph ───────────────────────────────────────────────────────
class TechnicalState(TypedDict):
    question: str   # shared with parent
    answer: str     # shared with parent

async def technical_node(state: TechnicalState) -> dict:
    response = await llm_agent.ainvoke([
        SystemMessage(content="You are a technical support agent. Answer the user's technical question."),
        HumanMessage(content=state["question"]),
    ])
    return {"answer": response.content}

technical_graph = StateGraph(TechnicalState)
technical_graph.add_node("technical", technical_node)
technical_graph.add_edge(START, "technical")
technical_graph.add_edge("technical", END)
technical_agent = technical_graph.compile()


# ─── Parent Graph (Orchestrator) ──────────────────────────────────────────────
def router(state: ParentState) -> Literal["billing", "technical"]:
    return state["category"]

parent_graph = StateGraph(ParentState)
parent_graph.add_node("classifier", classifier_agent)
parent_graph.add_node("billing",    billing_agent)
parent_graph.add_node("technical",  technical_agent)

parent_graph.add_edge(START, "classifier")
parent_graph.add_conditional_edges("classifier", router)
parent_graph.add_edge("billing", END)
parent_graph.add_edge("technical", END)

parent_compiled = parent_graph.compile()


# ─── Test ─────────────────────────────────────────────────────────────────────
result = await parent_compiled.ainvoke({"question": "My internet connection is very slow, what should I do?"})
print("Category:", result["category"])
print("Answer:  ", result["answer"])
